# 🚗 黄色胶带循迹 🐷
把黄色胶带贴在地上，小车会自动跟着走！

⚠️ **先在地板上贴一条黄色胶带再运行！**
按 **Q键** 退出

In [ ]:
import cv2
import numpy as np
import json
import serial
import time

# ─── 连接小车 ───
try:
    ser = serial.Serial('/dev/ttyAMA0', 115200, timeout=1)
    print('✅ 串口 /dev/ttyAMA0 连接成功！')
except:
    ser = serial.Serial('/dev/serial0', 115200, timeout=1)
    print('✅ 串口 /dev/serial0 连接成功！')

def cmd(data):
    ser.write((json.dumps(data) + '\n').encode('utf-8'))

def stop():
    cmd({'T':1, 'L':0, 'R':0})

In [ ]:
# ─── 摄像头 ───
try:
    from picamera2 import Picamera2
    picam = Picamera2()
    picam.configure(picam.create_preview_configuration(main={'size': (640, 480)}))
    picam.start()
    print('✅ CSI摄像头启动成功！')
    USING_CSI = True
except:
    cap = cv2.VideoCapture(0)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
    print('✅ USB摄像头启动成功！')
    USING_CSI = False

# ─── 黄色HSV参数 ───
YELLOW_LOWER = np.array([20, 80, 80])
YELLOW_UPPER = np.array([35, 255, 255])

BASE_SPEED = 40
FRAME_CENTER = 320
DEAD_ZONE = 40
TURN_SPEED = 30
MAX_TURN = 60

In [ ]:
print('''╔══════════════════════════════╗')
print('║  🐷 黄色循迹启动！            ║')
print('║  小车会自动追踪黄色胶带！     ║')
print('║  按 Q 退出                   ║')
print('╚══════════════════════════════╝')

try:
    while True:
        # 获取画面
        if USING_CSI:
            frame = picam.capture_array()
        else:
            ret, frame = cap.read()
            if not ret:
                continue

        # HSV + 黄色掩膜
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        mask = cv2.inRange(hsv, YELLOW_LOWER, YELLOW_UPPER)
        mask = cv2.erode(mask, None, iterations=2)
        mask = cv2.dilate(mask, None, iterations=2)

        # 找最大黄色区域
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        if contours:
            largest = max(contours, key=cv2.contourArea)
            if cv2.contourArea(largest) > 500:
                M = cv2.moments(largest)
                if M['m00']:
                    cx = int(M['m10'] / M['m00'])
                    offset = cx - FRAME_CENTER

                    cv2.drawContours(frame, [largest], -1, (0, 255, 255), 2)
                    cv2.circle(frame, (cx, 240), 5, (0, 0, 255), -1)

                    if abs(offset) < DEAD_ZONE:
                        cmd({'T':1, 'L':BASE_SPEED, 'R':BASE_SPEED})
                        s = '⬆️ 直走'
                    elif offset > 0:
                        t = min(abs(offset)/FRAME_CENTER, 1.0)
                        turn = int(TURN_SPEED + t*(MAX_TURN-TURN_SPEED))
                        cmd({'T':1, 'L':-turn, 'R':turn})
                        s = f'⬅️ 左转'
                    else:
                        t = min(abs(offset)/FRAME_CENTER, 1.0)
                        turn = int(TURN_SPEED + t*(MAX_TURN-TURN_SPEED))
                        cmd({'T':1, 'L':turn, 'R':-turn})
                        s = f'➡️ 右转'
                    cv2.putText(frame, s, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)
                    cv2.line(frame, (FRAME_CENTER, 0), (FRAME_CENTER, 480), (0,255,0), 1)
            else:
                stop()
                cv2.putText(frame, '⏸️ 没看到黄色', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,255), 2)
        else:
            stop()
            cv2.putText(frame, '⏸️ 没看到黄色', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,255), 2)

        cv2.imshow('🐷 黄色循迹', frame)
        cv2.imshow('黄色掩膜', mask)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

except KeyboardInterrupt:
    pass
except Exception as e:
    print(f'❌ 错误: {e}')
finally:
    stop()
    cmd({'T':133, 'X':0, 'Y':0, 'SPD':30, 'ACC':10})
    ser.close()
    if USING_CSI:
        picam.stop()
    else:
        cap.release()
    cv2.destroyAllWindows()
    print('✅ 安全停车')